In [136]:
import arcpy
import os

arcpy.env.overwriteOutput = True

#---------------------------------------------------------
# ArcGIS Pro project + workspace
#---------------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
workspace_gdb = aprx.defaultGeodatabase
arcpy.env.workspace = workspace_gdb

#---------------------------------------------------------
# Core dataset for presentation mapping
#---------------------------------------------------------
composite_fc = "PA_substations_composite_final"

#---------------------------------------------------------
# Presentation map names
#---------------------------------------------------------
map_floodplain_name = "Present_Floodplain"
map_bfe_name = "Present_Below_BFE"
map_overlay_name = "Present_Floodplain_BFE_Overlay"

#---------------------------------------------------------
# Basic checks
#---------------------------------------------------------
print("Project file:", aprx.filePath)
print("Default geodatabase:", workspace_gdb)
print("Workspace set to:", arcpy.env.workspace)
print("Composite feature class exists:", arcpy.Exists(composite_fc))

if arcpy.Exists(composite_fc):
    desc = arcpy.Describe(composite_fc)
    print("Shape type:", desc.shapeType)
    print("Spatial reference:", desc.spatialReference.name)
    print("Feature class path:", desc.catalogPath)

    n_rows = int(arcpy.management.GetCount(composite_fc)[0])
    print("Row count:", n_rows)
else:
    raise ValueError(f"Feature class not found: {composite_fc}")

#---------------------------------------------------------
# Optional: confirm key fields exist
#---------------------------------------------------------
field_names = [field.name for field in arcpy.ListFields(composite_fc)]

key_fields = [
    "fld_join_count",
    "bfe_flag",
    "dem_minus_bfe",
    "q_sub_scaled",
    "match_type",
    "chosen_rule",
    "operator",
    "substation",
    "county_name"
]

print("\nKey field check:")
for field_name in key_fields:
    print(f"  {field_name}: {'present' if field_name in field_names else 'missing'}")

Project file: C:\Users\Owner\Documents\ArcGIS\Projects\PA Flood Risk GIS\PA Flood Risk GIS.aprx
Default geodatabase: C:\Users\Owner\Documents\ArcGIS\Projects\PA Flood Risk GIS\PA Flood Risk GIS.gdb
Workspace set to: C:\Users\Owner\Documents\ArcGIS\Projects\PA Flood Risk GIS\PA Flood Risk GIS.gdb
Composite feature class exists: True
Shape type: Point
Spatial reference: NAD_1983_2011_StatePlane_Pennsylvania_South_FIPS_3702_Ft_US
Feature class path: C:\Users\Owner\Documents\ArcGIS\Projects\PA Flood Risk GIS\PA Flood Risk GIS.gdb\PA_substations_composite_final
Row count: 2333

Key field check:
  fld_join_count: present
  bfe_flag: present
  dem_minus_bfe: present
  q_sub_scaled: present
  match_type: present
  chosen_rule: present
  operator: present
  substation: present
  county_name: present


In [137]:
#---------------------------------------------------------
# Create presentation copy for map-specific fields
#---------------------------------------------------------
present_fc = "PA_substations_present_core"

if arcpy.Exists(present_fc):
    arcpy.management.Delete(present_fc)

arcpy.management.CopyFeatures(composite_fc, present_fc)

print("Created presentation feature class:", present_fc)

#---------------------------------------------------------
# Add map fields
#---------------------------------------------------------
existing_fields = [field.name for field in arcpy.ListFields(present_fc)]

if "fema_flood_exp" not in existing_fields:
    arcpy.management.AddField(
        in_table=present_fc,
        field_name="fema_flood_exp",
        field_type="SHORT"
    )

if "below_bfe_exp" not in existing_fields:
    arcpy.management.AddField(
        in_table=present_fc,
        field_name="below_bfe_exp",
        field_type="SHORT"
    )

if "flood_bfe_combo" not in existing_fields:
    arcpy.management.AddField(
        in_table=present_fc,
        field_name="flood_bfe_combo",
        field_type="TEXT",
        field_length=30
    )

print("Added presentation fields.")

#---------------------------------------------------------
# Populate map fields
#---------------------------------------------------------
with arcpy.da.UpdateCursor(
    present_fc,
    ["fld_join_count", "bfe_flag", "fema_flood_exp", "below_bfe_exp", "flood_bfe_combo"]
) as cursor:
    for fld_join_count, bfe_flag, fema_flood_exp, below_bfe_exp, flood_bfe_combo in cursor:

        fema_val = 1 if (fld_join_count is not None and fld_join_count > 0) else 0
        bfe_val = 1 if bfe_flag == "below_bfe" else 0

        if fema_val == 1 and bfe_val == 1:
            combo_val = "Both"
        elif fema_val == 1 and bfe_val == 0:
            combo_val = "Floodplain only"
        elif fema_val == 0 and bfe_val == 1:
            combo_val = "Below BFE only"
        else:
            combo_val = "Neither"

        cursor.updateRow((fld_join_count, bfe_flag, fema_val, bfe_val, combo_val))

print("Populated presentation fields.")

#---------------------------------------------------------
# Quick validation summaries
#---------------------------------------------------------
print("\nFloodplain exposure counts:")
with arcpy.da.SearchCursor(present_fc, ["fema_flood_exp"]) as cursor:
    flood_counts = {0: 0, 1: 0}
    for row in cursor:
        flood_counts[row[0]] += 1
print(flood_counts)

print("\nBelow-BFE counts:")
with arcpy.da.SearchCursor(present_fc, ["below_bfe_exp"]) as cursor:
    bfe_counts = {0: 0, 1: 0}
    for row in cursor:
        bfe_counts[row[0]] += 1
print(bfe_counts)

print("\nCombined class counts:")
combo_counts = {}
with arcpy.da.SearchCursor(present_fc, ["flood_bfe_combo"]) as cursor:
    for row in cursor:
        combo_val = row[0]
        combo_counts[combo_val] = combo_counts.get(combo_val, 0) + 1
print(combo_counts)

Created presentation feature class: PA_substations_present_core
Added presentation fields.
Populated presentation fields.

Floodplain exposure counts:
{0: 2137, 1: 196}

Below-BFE counts:
{0: 2038, 1: 295}

Combined class counts:
{'Floodplain only': 178, 'Neither': 1860, 'Below BFE only': 277, 'Both': 18}


In [138]:
#---------------------------------------------------------
# Step 3: add presentation layer to Present_Floodplain map
#---------------------------------------------------------
present_fc = "PA_substations_present_core"
map_floodplain_name = "Present_Floodplain"

aprx = arcpy.mp.ArcGISProject("CURRENT")

# find target map
target_map = None
for map_obj in aprx.listMaps():
    if map_obj.name == map_floodplain_name:
        target_map = map_obj
        break

if target_map is None:
    raise ValueError(f"Map not found: {map_floodplain_name}")

print("Using map:", target_map.name)

# remove older copies of the same layer if present
for layer_obj in target_map.listLayers():
    if layer_obj.name in ["PA_substations_present_core", "Substations - Floodplain Presentation"]:
        target_map.removeLayer(layer_obj)

# add the feature class
present_fc_path = arcpy.Describe(present_fc).catalogPath
target_map.addDataFromPath(present_fc_path)

# rename the newly added layer
added_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == "PA_substations_present_core":
        added_layer = layer_obj
        break

if added_layer is None:
    raise ValueError("Layer was added but could not be found in the map.")

added_layer.name = "Substations - Floodplain Presentation"

print("Added and renamed layer.")

#---------------------------------------------------------
# Optional: zoom active view to the layer
# Only works if the target map is the active view in ArcGIS Pro
#---------------------------------------------------------
try:
    active_view = aprx.activeView
    if active_view and hasattr(active_view, "map") and active_view.map.name == map_floodplain_name:
        layer_extent = active_view.getLayerExtent(added_layer, False, True)
        active_view.camera.setExtent(layer_extent)
        print("Zoomed active view to layer extent.")
    else:
        print("Map layer added. Activate the Present_Floodplain map tab if you want auto-zoom to work.")
except Exception as exc:
    print("Layer added, but auto-zoom was skipped:", exc)

Using map: Present_Floodplain
Added and renamed layer.
Zoomed active view to layer extent.


In [139]:
#---------------------------------------------------------
# Step 4: create exposed-only feature class and add it to
# Present_Floodplain
#---------------------------------------------------------
import arcpy

present_fc = "PA_substations_present_core"
floodplain_only_fc = "PA_substations_present_floodplain_only"
map_floodplain_name = "Present_Floodplain"

#---------------------------------------------------------
# Create exposed-only feature class
#---------------------------------------------------------
if arcpy.Exists(floodplain_only_fc):
    arcpy.management.Delete(floodplain_only_fc)

arcpy.analysis.Select(
    in_features=present_fc,
    out_feature_class=floodplain_only_fc,
    where_clause="fema_flood_exp = 1"
)

print("Created:", floodplain_only_fc)
print("Selected count:", int(arcpy.management.GetCount(floodplain_only_fc)[0]))

#---------------------------------------------------------
# Open project + target map
#---------------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")

target_map = None
for map_obj in aprx.listMaps():
    if map_obj.name == map_floodplain_name:
        target_map = map_obj
        break

if target_map is None:
    raise ValueError(f"Map not found: {map_floodplain_name}")

#---------------------------------------------------------
# Remove prior substation presentation layers from this map
#---------------------------------------------------------
remove_names = [
    "Substations - Floodplain Presentation",
    "Substations - In FEMA Floodplain",
    "PA_substations_present_core",
    "PA_substations_present_floodplain_only"
]

for layer_obj in target_map.listLayers():
    if layer_obj.name in remove_names:
        target_map.removeLayer(layer_obj)

#---------------------------------------------------------
# Add exposed-only layer
#---------------------------------------------------------
floodplain_only_path = arcpy.Describe(floodplain_only_fc).catalogPath
target_map.addDataFromPath(floodplain_only_path)

added_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == floodplain_only_fc:
        added_layer = layer_obj
        break

if added_layer is None:
    raise ValueError("Exposed-only layer was added but could not be found.")

added_layer.name = "Substations - In FEMA Floodplain"

print("Added layer to map:", added_layer.name)

#---------------------------------------------------------
# Optional auto-zoom if this map is the active view
#---------------------------------------------------------
try:
    active_view = aprx.activeView
    if active_view and hasattr(active_view, "map") and active_view.map.name == map_floodplain_name:
        layer_extent = active_view.getLayerExtent(added_layer, False, True)
        active_view.camera.setExtent(layer_extent)
        print("Zoomed active view to exposed substations.")
    else:
        print("Activate the Present_Floodplain map tab and use Zoom To Layer if needed.")
except Exception as exc:
    print("Layer added, but auto-zoom was skipped:", exc)

Created: PA_substations_present_floodplain_only
Selected count: 196
Added layer to map: Substations - In FEMA Floodplain
Zoomed active view to exposed substations.


In [140]:
#---------------------------------------------------------
# Step 5: create named-rivers presentation layer and add it
# to Present_Floodplain
#---------------------------------------------------------
import arcpy

nhd_flowline_fc = "nhdflowline_pa_2272"
river_fc = "PA_major_rivers_present"
map_floodplain_name = "Present_Floodplain"

#---------------------------------------------------------
# Confirm source layer exists
#---------------------------------------------------------
if not arcpy.Exists(nhd_flowline_fc):
    raise ValueError(f"Source flowline feature class not found: {nhd_flowline_fc}")

field_names = [field.name for field in arcpy.ListFields(nhd_flowline_fc)]
print("Flowline fields available:")
print(field_names)

#---------------------------------------------------------
# Build a named-flowline query
# Prefer GNIS_NAME if present
#---------------------------------------------------------
if "GNIS_NAME" in field_names:
    river_query = "GNIS_NAME IS NOT NULL AND GNIS_NAME <> ''"
elif "gnis_name" in field_names:
    river_query = "gnis_name IS NOT NULL AND gnis_name <> ''"
else:
    raise ValueError("Could not find GNIS_NAME field in nhdflowline_pa_2272.")

#---------------------------------------------------------
# Create named-rivers feature class
#---------------------------------------------------------
if arcpy.Exists(river_fc):
    arcpy.management.Delete(river_fc)

arcpy.analysis.Select(
    in_features=nhd_flowline_fc,
    out_feature_class=river_fc,
    where_clause=river_query
)

print("Created river presentation layer:", river_fc)
print("River feature count:", int(arcpy.management.GetCount(river_fc)[0]))

#---------------------------------------------------------
# Add to Present_Floodplain map
#---------------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")

target_map = None
for map_obj in aprx.listMaps():
    if map_obj.name == map_floodplain_name:
        target_map = map_obj
        break

if target_map is None:
    raise ValueError(f"Map not found: {map_floodplain_name}")

# remove previous copy if present
for layer_obj in target_map.listLayers():
    if layer_obj.name in ["Major Rivers", river_fc]:
        target_map.removeLayer(layer_obj)

river_fc_path = arcpy.Describe(river_fc).catalogPath
target_map.addDataFromPath(river_fc_path)

river_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == river_fc:
        river_layer = layer_obj
        break

if river_layer is None:
    raise ValueError("River layer was added but could not be found.")

river_layer.name = "Major Rivers"

print("Added layer to map:", river_layer.name)

#---------------------------------------------------------
# Try to move rivers below substations in draw order
#---------------------------------------------------------
substation_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == "Substations - In FEMA Floodplain":
        substation_layer = layer_obj
        break

if substation_layer is not None:
    target_map.moveLayer(substation_layer, river_layer, "BEFORE")
    print("Moved Major Rivers below substation layer.")
else:
    print("Substation layer not found for ordering. Adjust draw order manually if needed.")

Flowline fields available:
['OBJECTID', 'Shape', 'statecode', 'permanent_identifier', 'fdate', 'resolution', 'gnis_id', 'gnis_name', 'lengthkm', 'reachcode', 'flowdir', 'wbarea_permanent_identifier', 'ftype', 'fcode', 'mainpath', 'innetwork', 'visibilityfilter', 'enabled', 'vpuid', 'nhdplusid', 'fmeasure', 'tmeasure', 'GlobalID', 'Shape_Length', 'totdasqkm', 'hydroseq', 'levelpathi']
Created river presentation layer: PA_major_rivers_present
River feature count: 78509
Added layer to map: Major Rivers
Moved Major Rivers below substation layer.


In [141]:
import arcpy

nhd_flowline_fc = "nhdflowline_pa_2272"

field_names = [field.name for field in arcpy.ListFields(nhd_flowline_fc)]
print("Flowline fields:")
print(field_names)

check_fields = []
for field_name in ["GNIS_NAME", "gnis_name", "FTYPE", "FType", "ftype", "FCODE", "FCode", "fcode"]:
    if field_name in field_names:
        check_fields.append(field_name)

print("\nFields to inspect:", check_fields)

if len(check_fields) == 0:
    raise ValueError("Could not find expected name/type fields in nhdflowline_pa_2272.")

# print a small sample of unique values for type fields
for field_name in check_fields:
    if field_name.lower() in ["ftype", "fcode"]:
        values = set()
        with arcpy.da.SearchCursor(nhd_flowline_fc, [field_name]) as cursor:
            for row in cursor:
                if row[0] is not None:
                    values.add(row[0])
                if len(values) >= 25:
                    break
        print(f"\nSample values for {field_name}:")
        print(sorted(values))

Flowline fields:
['OBJECTID', 'Shape', 'statecode', 'permanent_identifier', 'fdate', 'resolution', 'gnis_id', 'gnis_name', 'lengthkm', 'reachcode', 'flowdir', 'wbarea_permanent_identifier', 'ftype', 'fcode', 'mainpath', 'innetwork', 'visibilityfilter', 'enabled', 'vpuid', 'nhdplusid', 'fmeasure', 'tmeasure', 'GlobalID', 'Shape_Length', 'totdasqkm', 'hydroseq', 'levelpathi']

Fields to inspect: ['gnis_name', 'ftype', 'fcode']

Sample values for ftype:
[334, 336, 420, 428, 460, 558]

Sample values for fcode:
[33400, 33600, 33601, 33603, 42000, 42003, 42801, 42802, 42803, 42804, 42807, 42811, 42820, 46000, 46003, 46006, 55800]


In [142]:
import arcpy

nhd_flowline_fc = "nhdflowline_pa_2272"
river_fc = "PA_major_rivers_present"
map_floodplain_name = "Present_Floodplain"

#---------------------------------------------------------
# Build a stricter major-rivers layer
#---------------------------------------------------------
river_query = "gnis_name IS NOT NULL AND gnis_name <> '' AND ftype = 460"

if arcpy.Exists(river_fc):
    arcpy.management.Delete(river_fc)

arcpy.analysis.Select(
    in_features=nhd_flowline_fc,
    out_feature_class=river_fc,
    where_clause=river_query
)

print("Created:", river_fc)
print("Feature count:", int(arcpy.management.GetCount(river_fc)[0]))

#---------------------------------------------------------
# Add it to the presentation map
#---------------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")

target_map = None
for map_obj in aprx.listMaps():
    if map_obj.name == map_floodplain_name:
        target_map = map_obj
        break

if target_map is None:
    raise ValueError(f"Map not found: {map_floodplain_name}")

for layer_obj in target_map.listLayers():
    if layer_obj.name in ["Major Rivers", "PA_major_rivers_present"]:
        target_map.removeLayer(layer_obj)

river_fc_path = arcpy.Describe(river_fc).catalogPath
target_map.addDataFromPath(river_fc_path)

river_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == river_fc:
        river_layer = layer_obj
        break

if river_layer is None:
    raise ValueError("River layer was added but could not be found.")

river_layer.name = "Major Rivers"

substation_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == "Substations - In FEMA Floodplain":
        substation_layer = layer_obj
        break

if substation_layer is not None:
    target_map.moveLayer(substation_layer, river_layer, "BEFORE")

print("Added stricter major-rivers layer to Present_Floodplain.")

Created: PA_major_rivers_present
Feature count: 48736
Added stricter major-rivers layer to Present_Floodplain.


In [143]:
import arcpy

nhd_area_fc = "nhdarea_pa"

if not arcpy.Exists(nhd_area_fc):
    raise ValueError(f"Feature class not found: {nhd_area_fc}")

field_names = [field.name for field in arcpy.ListFields(nhd_area_fc)]
print("nhdarea_pa fields:")
print(field_names)

check_fields = []
for field_name in [
    "gnis_name", "GNIS_NAME",
    "ftype", "FTYPE",
    "fcode", "FCODE",
    "areasqkm", "AreaSqKm", "areasqkm",
    "Shape_Area", "SHAPE_Area", "shape_area"
]:
    if field_name in field_names:
        check_fields.append(field_name)

print("\nFields to inspect:", check_fields)

for field_name in check_fields:
    if field_name.lower() in ["ftype", "fcode"]:
        values = set()
        with arcpy.da.SearchCursor(nhd_area_fc, [field_name]) as cursor:
            for row in cursor:
                if row[0] is not None:
                    values.add(row[0])
                if len(values) >= 25:
                    break
        print(f"\nSample values for {field_name}:")
        print(sorted(values))

nhdarea_pa fields:
['OBJECTID', 'Shape', 'Shape_Length', 'Shape_Area', 'statecode', 'permanent_identifier', 'fdate', 'resolution', 'gnis_id', 'gnis_name', 'areasqkm', 'elevation', 'ftype', 'fcode', 'visibilityfilter', 'nhdplusid', 'vpuid', 'GlobalID']

Fields to inspect: ['gnis_name', 'ftype', 'fcode', 'areasqkm', 'areasqkm', 'Shape_Area']

Sample values for ftype:
[307, 336, 343, 364, 398, 403, 431, 455, 460, 461]

Sample values for fcode:
[30700, 33600, 34305, 34306, 36400, 39800, 40307, 40308, 40309, 43100, 45500, 46000, 46003, 46006, 46100]


In [144]:
import arcpy

nhd_area_fc = "nhdarea_pa"
hydro_poly_fc = "PA_major_water_polygons_present"
map_floodplain_name = "Present_Floodplain"

# First-pass filter:
# - named features only
# - river-related area polygons
# - remove tiny polygons
hydro_query = (
    "gnis_name IS NOT NULL AND gnis_name <> '' "
    "AND ftype IN (460, 461) "
    "AND areasqkm >= 0.05"
)

if arcpy.Exists(hydro_poly_fc):
    arcpy.management.Delete(hydro_poly_fc)

arcpy.analysis.Select(
    in_features=nhd_area_fc,
    out_feature_class=hydro_poly_fc,
    where_clause=hydro_query
)

print("Created:", hydro_poly_fc)
print("Feature count:", int(arcpy.management.GetCount(hydro_poly_fc)[0]))

aprx = arcpy.mp.ArcGISProject("CURRENT")

target_map = None
for map_obj in aprx.listMaps():
    if map_obj.name == map_floodplain_name:
        target_map = map_obj
        break

if target_map is None:
    raise ValueError(f"Map not found: {map_floodplain_name}")

# Remove prior hydro reference layers from this map
remove_names = [
    "Major Rivers",
    "PA_major_rivers_present",
    "PA_major_rivers_present_long",
    "PA_major_water_polygons_present",
    "Hydro Areas"
]

for layer_obj in target_map.listLayers():
    if layer_obj.name in remove_names:
        target_map.removeLayer(layer_obj)

hydro_poly_path = arcpy.Describe(hydro_poly_fc).catalogPath
target_map.addDataFromPath(hydro_poly_path)

hydro_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == hydro_poly_fc:
        hydro_layer = layer_obj
        break

if hydro_layer is None:
    raise ValueError("Hydro polygon layer was added but could not be found.")

hydro_layer.name = "Hydro Areas"

# Put hydro polygons below substations
substation_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == "Substations - In FEMA Floodplain":
        substation_layer = layer_obj
        break

if substation_layer is not None:
    target_map.moveLayer(substation_layer, hydro_layer, "BEFORE")

print("Added Hydro Areas layer to Present_Floodplain.")

Created: PA_major_water_polygons_present
Feature count: 0
Added Hydro Areas layer to Present_Floodplain.


## Below Base Flood Elevation

In [145]:
import arcpy

# ---------------------------------------------------------
# Inputs / names
# ---------------------------------------------------------
present_fc = "PA_substations_present_core"
below_bfe_fc = "PA_substations_present_below_bfe_only"
map_bfe_name = "Present_Below_BFE"
hydro_ref_fc = "nhdarea_pa"

# ---------------------------------------------------------
# Create permanent below-BFE-only feature class
# ---------------------------------------------------------
if arcpy.Exists(below_bfe_fc):
    arcpy.management.Delete(below_bfe_fc)

arcpy.analysis.Select(
    in_features=present_fc,
    out_feature_class=below_bfe_fc,
    where_clause="below_bfe_exp = 1"
)

print("Created:", below_bfe_fc)
print("Selected count:", int(arcpy.management.GetCount(below_bfe_fc)[0]))

# ---------------------------------------------------------
# Open current project
# ---------------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")

# ---------------------------------------------------------
# Find or create the target map
# ---------------------------------------------------------
target_map = None
for map_obj in aprx.listMaps():
    if map_obj.name == map_bfe_name:
        target_map = map_obj
        break

if target_map is None:
    target_map = aprx.createMap(map_bfe_name, "MAP")
    print("Created new map:", map_bfe_name)
else:
    print("Using existing map:", map_bfe_name)

# ---------------------------------------------------------
# Remove prior copies of likely presentation layers
# ---------------------------------------------------------
remove_names = [
    "Substations - Below BFE",
    "PA_substations_present_below_bfe_only",
    "Hydro Areas",
    "nhdarea_pa"
]

for layer_obj in target_map.listLayers():
    if layer_obj.name in remove_names:
        target_map.removeLayer(layer_obj)

# ---------------------------------------------------------
# Add nhdarea_pa first (optional hydro reference)
# ---------------------------------------------------------
if arcpy.Exists(hydro_ref_fc):
    hydro_path = arcpy.Describe(hydro_ref_fc).catalogPath
    target_map.addDataFromPath(hydro_path)

    hydro_layer = None
    for layer_obj in target_map.listLayers():
        if layer_obj.name == hydro_ref_fc:
            hydro_layer = layer_obj
            break

    if hydro_layer is not None:
        hydro_layer.name = "Hydro Areas"
        print("Added hydro reference layer:", hydro_layer.name)
else:
    print("Hydro reference layer not found:", hydro_ref_fc)
    hydro_layer = None

# ---------------------------------------------------------
# Add below-BFE substation layer
# ---------------------------------------------------------
below_bfe_path = arcpy.Describe(below_bfe_fc).catalogPath
target_map.addDataFromPath(below_bfe_path)

bfe_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == below_bfe_fc:
        bfe_layer = layer_obj
        break

if bfe_layer is None:
    raise ValueError("Below-BFE layer was added but could not be found.")

bfe_layer.name = "Substations - Below BFE"
print("Added substation layer:", bfe_layer.name)

# ---------------------------------------------------------
# Put substations above hydro areas
# ---------------------------------------------------------
if hydro_layer is not None:
    target_map.moveLayer(bfe_layer, hydro_layer, "BEFORE")
    print("Moved Hydro Areas below the substation layer.")

# ---------------------------------------------------------
# Optional auto-zoom if this map is the active view
# ---------------------------------------------------------
try:
    active_view = aprx.activeView
    if active_view and hasattr(active_view, "map") and active_view.map.name == map_bfe_name:
        layer_extent = active_view.getLayerExtent(bfe_layer, False, True)
        active_view.camera.setExtent(layer_extent)
        print("Zoomed active view to the Below-BFE layer.")
    else:
        print("Activate the Present_Below_BFE tab and use Zoom To Layer if needed.")
except Exception as exc:
    print("Layer added, but auto-zoom was skipped:", exc)

Created: PA_substations_present_below_bfe_only
Selected count: 295
Using existing map: Present_Below_BFE
Hydro reference layer not found: nhdarea_pa
Added substation layer: Substations - Below BFE
Zoomed active view to the Below-BFE layer.


### Adding Legends

In [147]:
import arcpy

arcpy.env.overwriteOutput = True

present_fc = "PA_substations_present_core"

# ---------------------------------------------------------
# Constants
# ---------------------------------------------------------
total_substations = int(arcpy.management.GetCount(present_fc)[0])

# ---------------------------------------------------------
# Output tables
# ---------------------------------------------------------
floodplain_county_tbl = "tbl_present_floodplain_by_county"
below_bfe_county_tbl = "tbl_present_below_bfe_by_county"

# ---------------------------------------------------------
# Temporary filtered views
# ---------------------------------------------------------
floodplain_view = "vw_present_floodplain"
below_bfe_view = "vw_present_below_bfe"

# ---------------------------------------------------------
# Cleanup
# ---------------------------------------------------------
for table_name in [floodplain_county_tbl, below_bfe_county_tbl]:
    if arcpy.Exists(table_name):
        arcpy.management.Delete(table_name)

for view_name in [floodplain_view, below_bfe_view]:
    if arcpy.Exists(view_name):
        arcpy.management.Delete(view_name)

# ---------------------------------------------------------
# Create filtered table views
# ---------------------------------------------------------
arcpy.management.MakeTableView(
    in_table=present_fc,
    out_view=floodplain_view,
    where_clause="fema_flood_exp = 1"
)

arcpy.management.MakeTableView(
    in_table=present_fc,
    out_view=below_bfe_view,
    where_clause="below_bfe_exp = 1"
)

print("Filtered counts:")
print("  Floodplain:", int(arcpy.management.GetCount(floodplain_view)[0]))
print("  Below BFE:", int(arcpy.management.GetCount(below_bfe_view)[0]))

# ---------------------------------------------------------
# County summaries
# ---------------------------------------------------------
arcpy.analysis.Statistics(
    in_table=floodplain_view,
    out_table=floodplain_county_tbl,
    statistics_fields=[["OBJECTID", "COUNT"]],
    case_field="county_name"
)

arcpy.analysis.Statistics(
    in_table=below_bfe_view,
    out_table=below_bfe_county_tbl,
    statistics_fields=[["OBJECTID", "COUNT"]],
    case_field="county_name"
)

print("\nCreated county summary tables:")
print(" ", floodplain_county_tbl)
print(" ", below_bfe_county_tbl)

# ---------------------------------------------------------
# Totals and percentages
# ---------------------------------------------------------
floodplain_total = int(arcpy.management.GetCount(floodplain_view)[0])
below_bfe_total = int(arcpy.management.GetCount(below_bfe_view)[0])

floodplain_pct = 100.0 * floodplain_total / total_substations
below_bfe_pct = 100.0 * below_bfe_total / total_substations

print("\nOverall totals:")
print(f"Floodplain: {floodplain_total} of {total_substations} ({floodplain_pct:.1f}%)")
print(f"Below BFE: {below_bfe_total} of {total_substations} ({below_bfe_pct:.1f}%)")

# ---------------------------------------------------------
# Print top counties for quick layout use
# ---------------------------------------------------------
def print_top_counties(table_name, label):
    rows = []
    with arcpy.da.SearchCursor(table_name, ["county_name", "COUNT_OBJECTID"]) as cursor:
        for county_name, count_value in cursor:
            rows.append((county_name, count_value))

    rows = sorted(rows, key=lambda x: (-x[1], x[0]))

    print(f"\nTop counties for {label}:")
    for county_name, count_value in rows[:10]:
        print(f"  {county_name}: {count_value}")

print_top_counties(floodplain_county_tbl, "Floodplain")
print_top_counties(below_bfe_county_tbl, "Below BFE")

Filtered counts:
  Floodplain: 196
  Below BFE: 295

Created county summary tables:
  tbl_present_floodplain_by_county
  tbl_present_below_bfe_by_county

Overall totals:
Floodplain: 196 of 2333 (8.4%)
Below BFE: 295 of 2333 (12.6%)

Top counties for Floodplain:
  Berks County, Pennsylvania, USA: 23
  Northampton County, Pennsylvania, USA: 19
  Montgomery County, Pennsylvania, USA: 17
  Lawrence County, Pennsylvania, USA: 11
  Lehigh County, Pennsylvania, USA: 11
  Philadelphia County, Pennsylvania, USA: 11
  Bucks County, Pennsylvania, USA: 8
  Erie County, Pennsylvania, USA: 7
  Lebanon County, Pennsylvania, USA: 5
  Washington County, Pennsylvania, USA: 5

Top counties for Below BFE:
  Erie County, Pennsylvania, USA: 21
  Philadelphia County, Pennsylvania, USA: 14
  Delaware County, Pennsylvania, USA: 13
  Berks County, Pennsylvania, USA: 12
  Allegheny County, Pennsylvania, USA: 11
  Lycoming County, Pennsylvania, USA: 11
  Luzerne County, Pennsylvania, USA: 10
  Westmoreland County

## Four Classifications

In [1]:
import arcpy

present_fc = "PA_substations_present_core"
class_field = "flood_bfe_4class"

# ---------------------------------------------------------
# Add field if needed
# ---------------------------------------------------------
existing_fields = [field.name for field in arcpy.ListFields(present_fc)]

if class_field not in existing_fields:
    arcpy.management.AddField(
        in_table=present_fc,
        field_name=class_field,
        field_type="TEXT",
        field_length=50
    )
    print("Added field:", class_field)
else:
    print("Field already exists:", class_field)

# ---------------------------------------------------------
# Populate field
# ---------------------------------------------------------
with arcpy.da.UpdateCursor(
    present_fc,
    ["fema_flood_exp", "below_bfe_exp", class_field]
) as cursor:
    for fema_val, bfe_val, class_val in cursor:

        if fema_val == 0 and bfe_val == 0:
            new_class = "Out of floodplain and above BFE"
        elif fema_val == 1 and bfe_val == 0:
            new_class = "In floodplain but above BFE"
        elif fema_val == 0 and bfe_val == 1:
            new_class = "Not in floodplain but below BFE"
        elif fema_val == 1 and bfe_val == 1:
            new_class = "In floodplain and below BFE"
        else:
            new_class = "Unclassified"

        cursor.updateRow((fema_val, bfe_val, new_class))

print("Populated field:", class_field)

# ---------------------------------------------------------
# Quick count check
# ---------------------------------------------------------
class_counts = {}
with arcpy.da.SearchCursor(present_fc, [class_field]) as cursor:
    for row in cursor:
        val = row[0]
        class_counts[val] = class_counts.get(val, 0) + 1

print("\nFour-class counts:")
for key, value in sorted(class_counts.items()):
    print(f"{key}: {value}")

Added field: flood_bfe_4class
Populated field: flood_bfe_4class

Four-class counts:
In floodplain and below BFE: 18
In floodplain but above BFE: 178
Not in floodplain but below BFE: 277
Out of floodplain and above BFE: 1860


In [2]:
import arcpy

present_fc = "PA_substations_present_core"
map_name = "Present_Floodplain_BFE_4Class"

aprx = arcpy.mp.ArcGISProject("CURRENT")

# ---------------------------------------------------------
# Find or create the target map
# ---------------------------------------------------------
target_map = None
for map_obj in aprx.listMaps():
    if map_obj.name == map_name:
        target_map = map_obj
        break

if target_map is None:
    target_map = aprx.createMap(map_name, "MAP")
    print("Created new map:", map_name)
else:
    print("Using existing map:", map_name)

# ---------------------------------------------------------
# Remove older copies of likely layers
# ---------------------------------------------------------
remove_names = [
    "Substations - Floodplain/BFE 4-Class",
    "PA_substations_present_core",
    "Hydro Areas",
    "nhdarea_pa"
]

for layer_obj in target_map.listLayers():
    if layer_obj.name in remove_names:
        target_map.removeLayer(layer_obj)

# ---------------------------------------------------------
# Add nhdarea_pa as hydro reference
# ---------------------------------------------------------
hydro_layer = None
if arcpy.Exists("nhdarea_pa"):
    hydro_path = arcpy.Describe("nhdarea_pa").catalogPath
    target_map.addDataFromPath(hydro_path)

    for layer_obj in target_map.listLayers():
        if layer_obj.name == "nhdarea_pa":
            hydro_layer = layer_obj
            break

    if hydro_layer is not None:
        hydro_layer.name = "Hydro Areas"
        print("Added hydro reference layer.")
else:
    print("nhdarea_pa not found. Proceeding without hydro reference.")

# ---------------------------------------------------------
# Add the classified substation layer
# ---------------------------------------------------------
present_path = arcpy.Describe(present_fc).catalogPath
target_map.addDataFromPath(present_path)

substation_layer = None
for layer_obj in target_map.listLayers():
    if layer_obj.name == present_fc:
        substation_layer = layer_obj
        break

if substation_layer is None:
    raise ValueError("Could not find the added substation layer in the map.")

substation_layer.name = "Substations - Floodplain/BFE 4-Class"
print("Added classified substation layer.")

# ---------------------------------------------------------
# Put substations above hydro areas
# ---------------------------------------------------------
if hydro_layer is not None:
    target_map.moveLayer(substation_layer, hydro_layer, "BEFORE")
    print("Moved Hydro Areas below the substation layer.")

# ---------------------------------------------------------
# Optional auto-zoom if this map is active
# ---------------------------------------------------------
try:
    active_view = aprx.activeView
    if active_view and hasattr(active_view, "map") and active_view.map.name == map_name:
        layer_extent = active_view.getLayerExtent(substation_layer, False, True)
        active_view.camera.setExtent(layer_extent)
        print("Zoomed active view to layer extent.")
    else:
        print("Activate the map tab and use Zoom To Layer if needed.")
except Exception as exc:
    print("Layer added, but auto-zoom was skipped:", exc)

Created new map: Present_Floodplain_BFE_4Class
nhdarea_pa not found. Proceeding without hydro reference.
Added classified substation layer.
Activate the map tab and use Zoom To Layer if needed.
